In [ ]:
# Cell 1: Setup and Imports
import os
import re
import time
import random
import warnings
from collections import Counter
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')

# Check for GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Cell 2: Configuration
class Config:
    """Competition configuration."""
    
    # Model settings
    MODEL_NAME = "Qwen/Qwen2.5-Math-7B-Instruct"  # Change for larger models
    # Alternative models:
    # - "Qwen/Qwen2.5-Math-72B-Instruct" (requires more VRAM)
    # - "deepseek-ai/deepseek-math-7b-instruct"
    # - "meta-llama/Llama-3.1-8B-Instruct"
    
    # Quantization (for memory efficiency)
    LOAD_IN_4BIT = True
    LOAD_IN_8BIT = False
    
    # Generation settings
    MAX_NEW_TOKENS = 4096
    TEMPERATURE_MIN = 0.5
    TEMPERATURE_MAX = 1.0
    TOP_P = 0.95
    
    # Multi-sample reasoning
    NUM_SAMPLES = 8  # Generate multiple solutions per problem
    TIME_BUDGET = 300  # 5 minutes per problem
    
    # Majority voting
    USE_MAJORITY_VOTE = True
    
    # Answer range
    MIN_ANSWER = 0
    MAX_ANSWER = 99999

config = Config()
print("Configuration loaded!")

In [ ]:
# Cell 3: Load Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"Loading model: {config.MODEL_NAME}")

# Quantization config
bnb_config = None
if config.LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
elif config.LOAD_IN_8BIT:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.MODEL_NAME,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully!")

In [ ]:
# Cell 4: Answer Extraction Function
def extract_answer(solution: str) -> Optional[int]:
    """
    Extract the numerical answer from a solution.
    
    Looks for:
    1. \boxed{answer} format (preferred)
    2. "answer is X" patterns
    3. Last number in the solution
    """
    patterns = [
        r"\\boxed{(\d+)}",
        r"\\boxed\{(\d+)\}",
        r"boxed{(\d+)}",
        r"\$\\boxed{(\d+)}\$",
        r"answer is[:\s]+(\d+)",
        r"final answer[:\s]+(\d+)",
        r"= (\d+)$",
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, solution, re.IGNORECASE | re.MULTILINE)
        if matches:
            try:
                answer = int(matches[-1])
                if config.MIN_ANSWER <= answer <= config.MAX_ANSWER:
                    return answer
            except ValueError:
                continue
    
    # Fallback: last number in solution
    numbers = re.findall(r'\b(\d+)\b', solution)
    if numbers:
        try:
            answer = int(numbers[-1])
            if config.MIN_ANSWER <= answer <= config.MAX_ANSWER:
                return answer
        except ValueError:
            pass
    
    return None

# Test
test_solution = "Therefore, the answer is $\\boxed{42}$."
print(f"Test extraction: {extract_answer(test_solution)}")

In [ ]:
# Cell 5: Prompt Creation
def create_prompt(problem: str) -> str:
    """
    Create the prompt for the model.
    
    Uses appropriate format based on model type.
    """
    system_prompt = """You are a mathematical reasoning assistant competing in the AI Mathematical Olympiad.

Your task:
1. Solve the given problem step by step
2. Show all your work and reasoning clearly
3. Consider multiple approaches if helpful
4. Verify your answer before finalizing
5. Put your FINAL INTEGER ANSWER in \\boxed{}

CRITICAL: The answer MUST be a non-negative integer between 0 and 99999."""

    model_name_lower = config.MODEL_NAME.lower()
    
    if "qwen" in model_name_lower:
        return f"""<|im_start|>system
{system_prompt}<|im_end|>
<|im_start|>user
{problem}<|im_end|>
<|im_start|>assistant
Let me solve this step by step.

"""
    elif "llama" in model_name_lower:
        return f"""<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{problem} [/INST] Let me solve this step by step.

"""
    elif "mistral" in model_name_lower or "deepseek" in model_name_lower:
        return f"""<s>[INST] {system_prompt}

{problem} [/INST] Let me solve this step by step.

"""
    else:
        return f"""System: {system_prompt}

User: {problem}

Assistant: Let me solve this step by step.

"""

print("Prompt function created!")

In [ ]:
# Cell 6: Solution Generation
def generate_solution(problem: str, temperature: float = 0.7) -> Tuple[str, Optional[int]]:
    """
    Generate a single solution for a problem.
    
    Returns:
        Tuple of (solution_text, extracted_answer)
    """
    prompt = create_prompt(problem)
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=config.MAX_NEW_TOKENS,
            temperature=temperature,
            top_p=config.TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    solution = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    
    answer = extract_answer(solution)
    
    return solution, answer

print("Solution generator ready!")

In [ ]:
# Cell 7: Multi-Sample Solver with Majority Voting
def solve_problem(
    problem: str,
    num_samples: int = None,
    time_budget: int = None
) -> int:
    """
    Solve a problem using multi-sample reasoning and majority voting.
    
    Args:
        problem: The math problem text
        num_samples: Number of solution attempts
        time_budget: Maximum time in seconds
    
    Returns:
        Final integer answer
    """
    num_samples = num_samples or config.NUM_SAMPLES
    time_budget = time_budget or config.TIME_BUDGET
    
    start_time = time.time()
    answers = []
    
    for i in range(num_samples):
        # Check time budget
        if time.time() - start_time > time_budget * 0.9:
            break
        
        # Vary temperature for diversity
        temperature = random.uniform(config.TEMPERATURE_MIN, config.TEMPERATURE_MAX)
        
        try:
            solution, answer = generate_solution(problem, temperature)
            if answer is not None:
                answers.append(answer)
                print(f"  Sample {i+1}: Answer = {answer}")
        except Exception as e:
            print(f"  Sample {i+1}: Error - {e}")
            continue
    
    if not answers:
        print("  No valid answers found, returning 0")
        return 0
    
    # Majority voting
    if config.USE_MAJORITY_VOTE and len(answers) > 1:
        counter = Counter(answers)
        final_answer = counter.most_common(1)[0][0]
        vote_count = counter.most_common(1)[0][1]
        print(f"  Majority vote: {final_answer} ({vote_count}/{len(answers)} votes)")
    else:
        final_answer = answers[0]
        print(f"  Single answer: {final_answer}")
    
    return final_answer

print("Solver ready!")

In [ ]:
# Cell 8: Test on Sample Problem
test_problem = """
What is the remainder when $2^{100}$ is divided by 7?
"""

print("Testing on sample problem...")
print(f"Problem: {test_problem.strip()}")
print()

answer = solve_problem(test_problem, num_samples=3)  # Fewer samples for testing
print(f"\nFinal Answer: {answer}")

# Correct answer is 2 (2^3 = 1 mod 7, so 2^100 = 2^(3*33+1) = 2^1 = 2 mod 7)

In [ ]:
# Cell 9: Competition Submission Function
def solution_generator(problem: str) -> int:
    """
    Main function for Kaggle submission.
    
    This function is called by the Kaggle API for each problem.
    
    Args:
        problem: LaTeX-formatted math problem
    
    Returns:
        Integer answer between 0 and 99999
    """
    try:
        answer = solve_problem(problem)
        # Ensure answer is in valid range
        answer = max(config.MIN_ANSWER, min(config.MAX_ANSWER, answer))
        return int(answer)
    except Exception as e:
        print(f"Error solving problem: {e}")
        return 0

print("Submission function ready!")

In [ ]:
# Cell 10: Run Competition (Kaggle API Integration)
# Uncomment and run this cell when submitting to Kaggle

# from kaggle_environments import make
# 
# env = make("aimo-progress-prize-3")
# 
# # Run the environment
# observations = env.reset()
# 
# predictions = []
# for i, obs in enumerate(observations):
#     if obs is None:
#         continue
#     
#     problem = obs['question']
#     print(f"\n{'='*50}")
#     print(f"Problem {i+1}:")
#     print(problem[:200] + "..." if len(problem) > 200 else problem)
#     
#     answer = solution_generator(problem)
#     predictions.append(answer)
#     
#     print(f"Answer: {answer}")
# 
# print(f"\nTotal predictions: {len(predictions)}")

print("Competition cell ready - uncomment to run")

## Notes for Competition Submission

### To submit on Kaggle:

1. **Fork this notebook** in Kaggle

2. **Enable GPU** (preferably T4 x2 or P100)

3. **Add required datasets:**
   - AIMO Progress Prize 3 competition data
   - Your model weights (if fine-tuned)

4. **Uncomment Cell 10** to run the competition

5. **Submit notebook**

### Optimization Tips:

- **More samples = better accuracy** but slower
- Adjust `NUM_SAMPLES` based on time constraints
- Use larger models if VRAM permits
- Fine-tuned models perform better than base models